# Snippet from Cookbook.md


In [ ]:
#!/usr/bin/env python3
from compitum import Router, Config
from typing import Dict, Any, List
import json
class RoutingPipeline:
    def __init__(self, config_path: str):
        self.config = Config.from_yaml(config_path)
        self.router = Router(self.config)
        self.stages = []
    def route_with_validation(self, prompt: str) -> Dict[str, Any]:
        print("Stage 1: Initial Routing...")
        result = self.router.route(prompt)
        cert = result.certificate
        validations = {
            'energy_descent': cert['drift_signal'] < 0,
            'feasible': cert['feasible'],
            'high_entropy': cert['entropy'] > 1.0,
            'no_violations': all(c['slack'] >= 0 for c in cert['constraints'].values())
        }
        if not all(validations.values()):
            print("Validation failed:", {k for k, v in validations.items() if not v})
            return self._refine_route(prompt, cert, validations)
        print("Validation passed")
        return {'stage': 'initial', 'result': result, 'certificate': cert}
    def _refine_route(self, prompt: str, cert: Dict, validations: Dict) -> Dict[str, Any]:
        print("Stage 2: Refining...")
        tight_constraints = {
            name: data['slack']
            for name, data in cert['constraints'].items()
            if data['slack'] < 0.1
        }
        if tight_constraints:
            print(f"Relaxing: {list(tight_constraints.keys())}")
            refined_config = self.config.copy()
            for name, slack in tight_constraints.items():
                if slack < 0:
                    refined_config.constraints[name] *= 1.3
            refined_router = Router(refined_config)
            result = refined_router.route(prompt)
            print("Refinement complete")
            return {'stage': 'refined', 'result': result, 'certificate': result.certificate}
        return {'stage': 'refinement_failed', 'certificate': cert}
    def add_context_enrichment(self, result: Dict[str, Any], context_sources: List[str]) -> Dict[str, Any]:
        print("Stage 3: Context Enrichment...")
        context = "\n\n".join([open(src).read() for src in context_sources])
        original_prompt = result['certificate']['prompt']
        enriched_prompt = f"{original_prompt}\n\nContext:\n{context}"
        enriched_result = self.router.route(enriched_prompt)
        print("Context added")
        return {
            'stage': 'enriched',
            'result': enriched_result,
            'certificate': enriched_result.certificate,
            'context_sources': context_sources
        }
    def finalize_with_checks(self, result: Dict[str, Any]) -> Dict[str, Any]:
        print("Stage 4: Finalization...")
        cert = result['certificate']
        quality_score = cert['utility']
        coherence = cert['utility_components'].get('coherence', 0)
        final_result = {
            **result,
            'quality_score': quality_score,
            'coherence': coherence,
            'passed_final_checks': quality_score > 7.0 and coherence > -1.5,
            'pipeline_complete': True
        }
        print(f"Pipeline complete - Quality: {quality_score:.2f}")
        return final_result
    def run(self, prompt: str, context_sources: List[str] = None) -> Dict[str, Any]:
        print(f"\nStarting Pipeline for: '{prompt[:60]}...'\n")
        stage1 = self.route_with_validation(prompt)
        self.stages.append(stage1)
        if context_sources:
            stage2 = self.add_context_enrichment(stage1, context_sources)
            self.stages.append(stage2)
        else:
            stage2 = stage1
        final = self.finalize_with_checks(stage2)
        self.stages.append(final)
        return final
    def export_pipeline_trace(self, output_path: str):
        with open(output_path, 'w') as f:
            json.dump({
                'stages': self.stages,
                'pipeline_metadata': {
                    'total_stages': len(self.stages),
                    'final_quality': self.stages[-1].get('quality_score'),
                    'config': self.config.to_dict()
                }
            }, f, indent=2)
        print(f"Pipeline trace saved to {output_path}")
if __name__ == "__main__":
    pipeline = RoutingPipeline('configs/production.yaml')
    result = pipeline.run(
        prompt="Explain machine learning for beginners",
        context_sources=['docs/intro_to_ml.md']
    )
    pipeline.export_pipeline_trace('pipeline_trace.json')
    print("\n" + "="*60)
    print("PIPELINE SUMMARY")
    print("="*60)
    for i, stage in enumerate(pipeline.stages, 1):
        print(f"Stage {i} ({stage['stage']}): ", end="")
        if 'certificate' in stage:
            cert = stage['certificate']
            print(f"Utility={cert['utility']:.2f}, Drift={cert['drift_signal']:.3f}")
